# PyNRPF v0.1.0 - Publication Tables CSV Export

This notebook creates three CSV tables for the paper:
1. Dataset train-test split summary (computed from dataset)
2. Model performance summary (loaded from metrics JSON)
3. Implementation result summary (2023, manual/anonymized examples)

In [1]:
# Environment and imports
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.io import load_yaml, req, ensure_dir, load_parquet

print('Python:', sys.version)
print('CWD:   ', Path.cwd())
print('REPO:  ', REPO_ROOT)

Python: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
CWD:    C:\Users\z5404477\Documents\PyNRPF\notebooks
REPO:   C:\Users\z5404477\Documents\PyNRPF


In [2]:
# Config and paths
CFG_PATH = REPO_ROOT / 'config' / 'run.yaml'
cfg = load_yaml(CFG_PATH)

RUN_TAG = str(req(cfg, 'run.run_tag'))

DATASET_PATH = (REPO_ROOT / str(req(cfg, 'paths.dataset_parquet'))).resolve()
OUTPUT_DIR = (REPO_ROOT / str(req(cfg, 'paths.output_dir'))).resolve()
TABLE_DIR = OUTPUT_DIR / 'publication_tables'
ensure_dir(TABLE_DIR)

METRICS_JSON = OUTPUT_DIR / f'metrics__{RUN_TAG}.json'

COL_SITE = str(req(cfg, 'data.columns.site'))
COL_TS = str(req(cfg, 'data.columns.ts'))
COL_GT = str(req(cfg, 'data.columns.gt'))

TRAIN_START = pd.Timestamp(str(req(cfg, 'split.train_start'))).date()
TRAIN_END = pd.Timestamp(str(req(cfg, 'split.train_end'))).date()
TEST_START = pd.Timestamp(str(req(cfg, 'split.test_start'))).date()
TEST_END = pd.Timestamp(str(req(cfg, 'split.test_end'))).date()

print('Config loaded from:', CFG_PATH)
print('Run tag:', RUN_TAG)
print('Dataset:', DATASET_PATH)
print('Metrics:', METRICS_JSON)
print('Output table dir:', TABLE_DIR)
print(f'Split: train {TRAIN_START}..{TRAIN_END} | test {TEST_START}..{TEST_END}')

Config loaded from: C:\Users\z5404477\Documents\PyNRPF\config\run.yaml
Run tag: local_dev
Dataset: C:\Users\z5404477\Documents\PyNRPF\data\raw\rpf_dataset.parquet
Metrics: C:\Users\z5404477\Documents\PyNRPF\outputs\metrics__local_dev.json
Output table dir: C:\Users\z5404477\Documents\PyNRPF\outputs\publication_tables
Split: train 2021-11-01..2023-09-30 | test 2023-10-01..2024-09-30


In [3]:
# Load dataset and split
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

if not METRICS_JSON.exists():
    raise FileNotFoundError(f'Metrics JSON not found: {METRICS_JSON}')

df = load_parquet(DATASET_PATH)
df[COL_TS] = pd.to_datetime(df[COL_TS], errors='raise')
df['date'] = df[COL_TS].dt.date

train_mask = (df['date'] >= TRAIN_START) & (df['date'] <= TRAIN_END)
test_mask = (df['date'] >= TEST_START) & (df['date'] <= TEST_END)

df_train = df.loc[train_mask].copy()
df_test = df.loc[test_mask].copy()
n_other = int((~train_mask & ~test_mask).sum())

print(f'Train rows: {len(df_train):,}')
print(f'Test rows:  {len(df_test):,}')
print(f'Other rows: {n_other:,}')

Loaded 1,011,264 rows x 5 cols from rpf_dataset.parquet


Train rows: 666,432
Test rows:  344,832
Other rows: 0


In [4]:
# Table 1 - Dataset train-test split summary
def build_split_rows(df_split: pd.DataFrame, split_name: str) -> list[dict]:
    interval_total = int(len(df_split))
    interval_need = int((df_split[COL_GT] < 0).sum())
    interval_pct = round((interval_need / interval_total * 100.0) if interval_total else 0.0, 3)

    day_df = (
        df_split.groupby([COL_SITE, 'date'])
        .agg(need_correction=(COL_GT, lambda s: bool((s < 0).any())))
        .reset_index()
    )
    day_total = int(len(day_df))
    day_need = int(day_df['need_correction'].sum())
    day_pct = round((day_need / day_total * 100.0) if day_total else 0.0, 3)

    return [
        {
            'split': split_name,
            'level': 'day',
            'n_total': day_total,
            'n_need_correction': day_need,
            'pct_need_correction': day_pct,
        },
        {
            'split': split_name,
            'level': 'interval',
            'n_total': interval_total,
            'n_need_correction': interval_need,
            'pct_need_correction': interval_pct,
        },
    ]

table1_rows = []
table1_rows.extend(build_split_rows(df_train, 'train'))
table1_rows.extend(build_split_rows(df_test, 'test'))

table1 = pd.DataFrame(table1_rows)
table1 = table1[['split', 'level', 'n_total', 'n_need_correction', 'pct_need_correction']]
table1

,split,level,n_total,n_need_correction,pct_need_correction
0,train,day,6986,2095,29.989
1,train,interval,666432,27678,4.153
2,test,day,3657,1328,36.314
3,test,interval,344832,20302,5.888


In [5]:
# Table 2 - Model performance summary (from metrics JSON, unpivoted split rows)
with METRICS_JSON.open('r', encoding='utf-8') as f:
    metrics = json.load(f)

method_name_map = {
    'm7_threshold': 'm7_dtr',
    'm8_xgb': 'm8_xgb',
}

def metric_rows(method_key: str, model_level: str, interval_scope: str, result_key: str) -> list[dict]:
    section = metrics[method_key][result_key]
    rows = []
    for split in ['train', 'test']:
        part = section[split]
        rows.append({
            'method': method_name_map.get(method_key, method_key),
            'model_level': model_level,
            'interval_scope': interval_scope,
            'split': split,
            'P': round(float(part['precision']), 3),
            'R': round(float(part['recall']), 3),
            'F1': round(float(part['f1']), 3),
        })
    return rows

base_specs = [
    ('m7_threshold', 'day', 'na', 'day'),
    ('m7_threshold', 'interval', 'all_days', 'interval_all_days'),
    ('m7_threshold', 'interval', 'tp_days_only', 'interval_tp_days_only'),
    ('m8_xgb', 'day', 'na', 'day'),
    ('m8_xgb', 'interval', 'all_days', 'interval_all_days'),
    ('m8_xgb', 'interval', 'tp_days_only', 'interval_tp_days_only'),
]

table2_rows = []
for split in ['train', 'test']:
    for method_key, model_level, interval_scope, result_key in base_specs:
        rows = metric_rows(method_key, model_level, interval_scope, result_key)
        table2_rows.extend([r for r in rows if r['split'] == split])

table2 = pd.DataFrame(table2_rows)
table2 = table2[[
    'method', 'model_level', 'interval_scope', 'split',
    'P', 'R', 'F1',
]]
table2

,method,model_level,interval_scope,split,P,R,F1
0,m7_dtr,day,na,train,0.889,0.980,0.932
1,m7_dtr,interval,all_days,train,0.807,0.770,0.788
2,m7_dtr,interval,tp_days_only,train,0.840,0.770,0.804
3,m8_xgb,day,na,train,1.000,1.000,1.000
4,m8_xgb,interval,all_days,train,0.967,0.838,0.898
5,m8_xgb,interval,tp_days_only,train,0.967,0.838,0.898
6,m7_dtr,day,na,test,0.912,0.963,0.937
7,m7_dtr,interval,all_days,test,0.844,0.796,0.819
8,m7_dtr,interval,tp_days_only,test,0.869,0.796,0.831
9,m8_xgb,day,na,test,0.954,0.958,0.956


In [6]:
# Table 3 - Implementation result (2023, manual/anonymized examples)
table3 = pd.DataFrame([
    {
        'site_name': 'network',
        'year': 2023,
        'min_demand_MW_before': 1481.0,
        'min_demand_MW_after': 1447.0,
        'ts_min_before': '',
        'ts_min_after': '',
    },
    {
        'site_name': 'Site_A',
        'year': 2023,
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -8.21,
        'ts_min_before': '25/10/2023 13:00',
        'ts_min_after': '21/11/2023 14:15',
    },
    {
        'site_name': 'Site_B',
        'year': 2023,
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -6.41,
        'ts_min_before': '17/05/2023 8:15',
        'ts_min_after': '5/10/2023 12:30',
    },
    {
        'site_name': 'Site_C',
        'year': 2023,
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -5.24,
        'ts_min_before': '30/11/2023 9:30',
        'ts_min_after': '6/03/2023 13:45',
    },
])

table3['min_demand_MW_correction'] = (
    table3['min_demand_MW_before'] - table3['min_demand_MW_after']
).round(2)

table3 = table3[[
    'site_name', 'year',
    'min_demand_MW_before', 'min_demand_MW_after', 'min_demand_MW_correction',
    'ts_min_before', 'ts_min_after',
]]
table3


,site_name,year,min_demand_MW_before,min_demand_MW_after,min_demand_MW_correction,ts_min_before,ts_min_after
0,network,2023,1481.0,1447.00,34.00,,
1,Site_A,2023,0.0,-8.21,8.21,25/10/2023 13:00,21/11/2023 14:15
2,Site_B,2023,0.0,-6.41,6.41,17/05/2023 8:15,5/10/2023 12:30
3,Site_C,2023,0.0,-5.24,5.24,30/11/2023 9:30,6/03/2023 13:45


In [7]:
# Export CSV files
TABLE1_PATH = TABLE_DIR / 'table1_dataset_split_summary.csv'
TABLE2_PATH = TABLE_DIR / 'table2_model_performance_summary.csv'
TABLE3_PATH = TABLE_DIR / 'table3_implementation_result_ausgrid_2023.csv'

def _safe_write_csv(df: pd.DataFrame, path: Path) -> Path:
    try:
        df.to_csv(path, index=False)
        return path
    except PermissionError:
        fallback = path.with_name(path.stem + '_updated.csv')
        df.to_csv(fallback, index=False)
        print(f'Permission denied for {path}; wrote to {fallback} instead')
        return fallback

TABLE1_WRITTEN = _safe_write_csv(table1, TABLE1_PATH)
TABLE2_WRITTEN = _safe_write_csv(table2, TABLE2_PATH)
TABLE3_WRITTEN = _safe_write_csv(table3, TABLE3_PATH)

print(f'Wrote: {TABLE1_WRITTEN}')
print(f'Wrote: {TABLE2_WRITTEN}')
print(f'Wrote: {TABLE3_WRITTEN}')

Wrote: C:\Users\z5404477\Documents\PyNRPF\outputs\publication_tables\table1_dataset_split_summary.csv
Wrote: C:\Users\z5404477\Documents\PyNRPF\outputs\publication_tables\table2_model_performance_summary.csv
Wrote: C:\Users\z5404477\Documents\PyNRPF\outputs\publication_tables\table3_implementation_result_ausgrid_2023.csv


In [8]:
# Validation checks
assert TABLE1_WRITTEN.exists(), 'Missing table1 CSV'
assert TABLE2_WRITTEN.exists(), 'Missing table2 CSV'
assert TABLE3_WRITTEN.exists(), 'Missing table3 CSV'

assert len(table1) == 4, f'Table 1 row count expected 4, got {len(table1)}'
assert len(table2) == 12, f'Table 2 row count expected 12, got {len(table2)}'
assert len(table3) == 4, f'Table 3 row count expected 4, got {len(table3)}'

req_t1_cols = ['split', 'level', 'n_total', 'n_need_correction', 'pct_need_correction']
assert table1[req_t1_cols].notna().all().all(), 'Table 1 has nulls in required columns'

metric_cols = ['P', 'R', 'F1']
assert table2[metric_cols].notna().all().all(), 'Table 2 has missing metrics'

assert set(table3['site_name']) == {'network', 'Site_A', 'Site_B', 'Site_C'}, 'Unexpected Table 3 site names'
assert (table3['year'] == 2023).all(), 'Table 3 year must be 2023'

print('Validation passed.')
print('Table 1 rows:', len(table1))
print('Table 2 rows:', len(table2))
print('Table 3 rows:', len(table3))

table1


Validation passed.
Table 1 rows: 4
Table 2 rows: 12
Table 3 rows: 4


,split,level,n_total,n_need_correction,pct_need_correction
0,train,day,6986,2095,29.989
1,train,interval,666432,27678,4.153
2,test,day,3657,1328,36.314
3,test,interval,344832,20302,5.888
